# Build the priors

Turns each downloaded prior into one of the two forms that [`pwpa.py`](pwpa.py) reads. PIE's five sources are all profiles.

- **profile**, `priors/profile/<source>.parquet`: one row per gene. The similarity of two genes is the dot product of their rows, so the rows are scaled first. A pair is covered when both genes have a row.
- **network**, `priors/network/<source>.parquet`: weighted gene pairs. A missing pair has similarity 0 and still counts as covered.

At fit time `pwpa.py` z-scores every prior over the training pairs, which puts all priors on a common scale.

In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.append("../utils")
from paths import data_dir

import pwpa

In [ ]:
def standard_rows(table):
    """Profile rows from a numeric gene x feature table (screens, DepMap, expression): each row centred and scaled to
    unit length, so the dot product of two rows is their Pearson correlation. A missing value counts as the row mean."""
    return unit_rows(table.sub(table.mean(axis=1), axis=0).fillna(0))


def unit_rows(X):
    """Rows scaled to unit length. A row of zeros carries no information, so its gene is dropped (left uncovered)."""
    norm = np.linalg.norm(X, axis=1)
    return X[norm > 0].div(norm[norm > 0], axis=0)


# TODO(orcd): check against how K031-pie-five-path builds its channels

In [ ]:
# TODO(orcd): check in K031-pie-five-path which sources are Pearson (standard_rows) and which cosine (unit_rows),
# and keep the gene identifiers that the processed datasets use.
#
# for source in ["perturbation_text", "ncbi_text", "esm2", "string_space", "depmap_gene_effect"]:
#     vectors = read data_dir("priors", "raw", source)   # genes x dimensions
#     standard_rows(vectors).to_parquet(data_dir("priors", "profile") / f"{source}.parquet")

In [ ]:
priors = pwpa.load_priors(data_dir("priors"))
print(len(priors), "priors:", ", ".join(priors))